> **归档说明**：本 notebook 记录项目开发过程中的中间实验，保留用于复盘和审计，不作为最终展示入口。部分输入可能依赖本地生成但未上传 GitHub 的过程产物，例如 `outputs/predictions/`、`threshold_metrics` 或 `trial_results`。复现这些历史实验前，请先查看 `reports/notebook_reproducibility_audit.md` 中对应的再生成脚本说明。项目最终展示入口见 `notebooks/final/` 和 `README.md`。
>
> 该实验结果仅作为历史对照，不作为最终模型选择依据。


# Day 15：Controlled XGBoost Tuning

本轮只在 official train 内部的 `train_inner / valid` 上做受控 XGBoost 调参。official test 不参与参数、特征方案或阈值选择，Day16 才能对少数 tuned 候选方案做最终观察。

## 为什么不是 GridSearch

全组合 GridSearch 在当前参数空间下成本过高，也容易把项目变成单纯调参。本轮采用 two-stage randomized search：先用 broad search 覆盖较大空间，再基于 valid 上较优 trial 构造 refined search space。排序仍以 valid total cost 为核心，辅助观察 FN、recall、F2 和 PR-AUC。

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd

from scania_aps.config import get_config
from scania_aps.data.load_data import load_train_test_with_target
from scania_aps.data.split_data import split_train_valid
from scania_aps.features.structural_feature_design import load_structural_feature_config
from scania_aps.models.xgb_tuning import run_xgb_tuning_experiments

cfg = get_config(PROJECT_ROOT / "config" / "config.yaml")
structural_config = load_structural_feature_config(PROJECT_ROOT / "config" / "structural_features.yaml")
cfg.xgb_tuning["candidate_strategies"]

['baseline_median_all',
 'median_all_structural_all',
 'drop_high_missing_median']

## 数据划分

本轮读取 official train/test，但只使用 official train 内部划分出的 `train_inner` 和 `valid`。official test 在本 notebook 中不参与调参。

In [2]:
train_df, _official_test_df = load_train_test_with_target(cfg)
train_inner_df, valid_df = split_train_valid(train_df, cfg)
train_inner_df.shape, valid_df.shape

((48000, 172), (12000, 172))

## 运行 Day15 调参

完整运行可直接使用脚本：

```powershell
python scripts/13_xgb_tuning_valid_experiments.py
```

如果只是 notebook 快速复盘，建议先读取脚本已经保存的输出文件，避免在 notebook 内重复训练大量 trial。

In [3]:
best_path = cfg.metrics_dir / "day15_xgb_tuning_valid_best_summary.csv"
trial_path = cfg.metrics_dir / "day15_xgb_tuning_valid_trial_results.csv"
refine_path = cfg.metrics_dir / "day15_xgb_tuning_refinement_summary.csv"

if best_path.exists():
    best_summary = pd.read_csv(best_path)
    trial_results = pd.read_csv(trial_path)
    refinement_summary = pd.read_csv(refine_path)
else:
    results = run_xgb_tuning_experiments(
        train_inner_df=train_inner_df,
        valid_df=valid_df,
        cfg=cfg,
        structural_config=structural_config,
        candidate_strategies=cfg.xgb_tuning["candidate_strategies"],
    )
    best_summary = results["tuning_best_summary"]
    trial_results = results["tuning_trial_results"]
    refinement_summary = results["tuning_refinement_summary"]

best_summary.head()

,selection_rank,global_trial_rank,model_name,strategy,threshold,precision,recall,f1,f2,average_precision,...,normalized_strategy,fit_dataset,evaluation_dataset,uses_official_test,n_original_features,n_structural_features,n_dropped_features,dropped_features,structural_feature_names,selection_dataset
0,1,1,xgboost_tuned,median_all_structural_all,0.31,0.442177,0.975,0.608424,0.785657,0.874469,...,median_all_structural_all,train_inner,valid,False,170,60,0,NaN,sample_missing_count|sample_missing_rate|sampl...,valid
1,2,2,xgboost_tuned,drop_high_missing_median,0.19,0.387352,0.980,0.555241,0.750383,0.864892,...,drop_high_missing_median,train_inner,valid,False,168,0,2,bq_000|br_000,NaN,valid
2,3,4,xgboost_tuned,baseline_median_all,0.13,0.407950,0.975,0.575221,0.762911,0.888664,...,baseline_median_all,train_inner,valid,False,170,0,0,NaN,NaN,valid


## Broad vs Refined

refined 阶段只能基于 broad 阶段在 valid 上的 Top trials 构造，不能使用 official test。若 refined 相比 broad 提升有限，需要如实记录，这可能意味着当前特征方案下 XGBoost 参数空间已经接近瓶颈。

In [4]:
refinement_summary

,candidate_strategy,broad_trials,refine_trials,top_trials_used_for_refinement,broad_best_trial_id,broad_best_total_cost,broad_best_fn,refined_best_trial_id,refined_best_total_cost,refined_best_fn,best_stage,best_trial_id,best_total_cost,refined_cost_delta_vs_broad
0,baseline_median_all,20,8,10,baseline_median_all_broad_016,5370,7,baseline_median_all_refined_002,5330,5,refined,baseline_median_all_refined_002,5330,-40
1,median_all_structural_all,20,8,10,median_all_structural_all_broad_009,4960,5,median_all_structural_all_refined_001,5770,4,broad,median_all_structural_all_broad_009,4960,810
2,drop_high_missing_median,20,8,10,drop_high_missing_median_broad_007,5100,4,drop_high_missing_median_refined_001,5310,4,broad,drop_high_missing_median_broad_007,5100,210


## Top Trials

下面只展示 valid 上 total cost 最低的若干 trial。valid 最优不是最终模型，只是 Day16 official test 观察的候选。

In [5]:
display_cols = [
    "candidate_strategy", "search_stage", "trial_id", "threshold",
    "precision", "recall", "f2", "average_precision", "fp", "fn", "total_cost",
]
trial_results[display_cols].head(20)

,candidate_strategy,search_stage,trial_id,threshold,precision,recall,f2,average_precision,fp,fn,total_cost
0,median_all_structural_all,broad,median_all_structural_all_broad_009,0.31,0.442177,0.975,0.785657,0.874469,246,5,4960
1,drop_high_missing_median,broad,drop_high_missing_median_broad_007,0.19,0.387352,0.980,0.750383,0.864892,310,4,5100
2,drop_high_missing_median,refined,drop_high_missing_median_refined_001,0.10,0.371917,0.980,0.738508,0.888637,331,4,5310
3,baseline_median_all,refined,baseline_median_all_refined_002,0.13,0.407950,0.975,0.762911,0.888664,283,5,5330
4,baseline_median_all,broad,baseline_median_all_broad_016,0.20,0.507895,0.965,0.817797,0.894212,187,7,5370
5,drop_high_missing_median,broad,drop_high_missing_median_broad_014,0.26,0.398773,0.975,0.756400,0.860598,294,5,5440
6,drop_high_missing_median,broad,drop_high_missing_median_broad_008,0.24,0.395538,0.975,0.754060,0.856480,298,5,5480
7,median_all_structural_all,broad,median_all_structural_all_broad_018,0.21,0.393145,0.975,0.752315,0.866574,301,5,5510
8,baseline_median_all,refined,baseline_median_all_refined_003,0.15,0.433036,0.970,0.777244,0.886140,254,6,5540
9,baseline_median_all,refined,baseline_median_all_refined_004,0.28,0.432071,0.970,0.776621,0.866637,255,6,5550


## Day15 小结模板

- 本轮没有使用 official test。
- 候选策略固定为 `baseline_median_all`、`median_all_structural_all` 和 `drop_high_missing_median`。
- 每个 trial 都在 valid 上遍历阈值，按 total cost、FN、recall、F2、PR-AUC 排序。
- Day16 只能选择 Day15 valid 上表现最稳的 1-2 个 tuned 方案做 official test 最终观察。
- 不解释匿名字段的真实物理含义，不做 SHAP / PCA / SVM / LightGBM / CatBoost。